# Basic Setting

In [ ]:
import os 
import pandas as pd
import numpy as np 
import torchaudio
from audiocraft.models import MusicGen
from audiocraft.data.audio import audio_write
from pydub import AudioSegment, effects  
import glob

path_to_ffmpeg = "/path/to/ffmpeg"
os.environ['PATH'] += os.pathsep + f"{path_to_ffmpeg}" # ffmpeg and ffmprobe binary file directory


model = MusicGen.get_pretrained('melody')
root_dir = "Revisiting-Your-Memory"
eeg_freq = 500  #Hz
target_freq = 32000 #Hz


#model.set_generation_params(duration=35)  # generate 8 seconds.

In [ ]:
subID = 'sub-01'
data_type = 'true' 
#data_type = 'fake'     # this is for fake (order perturbated) data 
emotion_table = pd.read_csv(os.path.join(*[root_dir, subID, f"{subID}-words.csv"]))
total_eeg_sec = len(emotion_table) / eeg_freq
# setting model parameter for generation
model.set_generation_params(duration=total_eeg_sec) 

# loading audio file and resampling and matching the total eeg length and music length 
orig_melody, sr = torchaudio.load(os.path.join(*[root_dir, subID, f"music/{subID}-melody.wav"]))
orig_melody = torchaudio.functional.resample(orig_melody, orig_freq=sr, new_freq=target_freq)
total_melody_sec = orig_melody.shape[-1] / target_freq
assert  total_melody_sec > total_eeg_sec
orig_melody = orig_melody[:, :int(total_eeg_sec*target_freq)]

In [ ]:
# generating three versions of musics 
emotion_labels = list(pd.unique(emotion_table['words'].dropna()))
emotion_labels_bin = []
for emotion in emotion_labels: 
    if np.argmax(emotion_table[emotion_table['words']==emotion][['neutral', 'positive','negative']].iloc[0].values) == 1: 
        emotion_labels_bin.append('pos')
    elif np.argmax(emotion_table[emotion_table['words']==emotion][['neutral', 'positive','negative']].iloc[0].values) == 2:
        emotion_labels_bin.append('neg') 

In [ ]:

### generating music 
base_prompt = "song with only grand piano solo play. No other instruments play."
# generating neutral music 
emotion = 'control'
prompt = f'A {emotion} {base_prompt}'
gen_wav = model.generate_with_chroma([prompt], orig_melody.expand(1, -1, -1), target_freq)
torchaudio.save(os.path.join(*[root_dir, subID, f"music/melody_{emotion}.wav"]), gen_wav.squeeze(0).detach().cpu(), target_freq)

# generating affect reflecting music 
for bin, emotion in zip(emotion_labels_bin, emotion_labels): 
    prompt = f'A {emotion} {base_prompt}'
    gen_wav = model.generate_with_chroma([prompt], orig_melody.expand(1, -1, -1), target_freq)
    torchaudio.save(os.path.join(*[root_dir, subID, f"music/melody_{emotion}.wav"]), gen_wav.squeeze(0).detach().cpu(), target_freq)

In [ ]:
### mixing music 
# seperating chunks 
start_point = [0]
end_point = [] 

for i in range(len(emotion_table)): 
    if len(pd.unique(emotion_table.iloc[i:i+2]['words'])) == 2: 
        start_point.append(i+1)
        end_point.append(i)

end_point.append(len(emotion_table))
        
emotion_chunk_label = []
for i in start_point: 
    emotion_chunk_label.append(emotion_table.iloc[i]['words'])

# loading music files
music_files = {} 
orig_melody, sr = torchaudio.load(os.path.join(*[root_dir, subID, "music/melody.wav"]))
orig_melody = torchaudio.functional.resample(orig_melody, orig_freq=sr, new_freq=target_freq)
music_files['neutral'] = orig_melody
for bin, emotion in zip(emotion_labels_bin, emotion_labels): 
    new_melody, sr = torchaudio.load(os.path.join(*[root_dir, subID, f"music/melody_{emotion}.wav"]))
    new_melody = torchaudio.functional.resample(new_melody, orig_freq=sr, new_freq=target_freq)
    music_files[emotion] = new_melody

In [ ]:
# rearrange each emotion reflecting musics while temporally synchronizing emotion label from EEG 
if os.path.exists(os.path.join(*[root_dir, subID, f"music/{data_type}/melody_chunks"])) is False: 
    os.mkdir(os.path.join(*[root_dir, subID, f"music/{data_type}/melody_chunks"]))

for i, (start, end, emotion_chunk) in enumerate(zip(start_point, end_point, emotion_chunk_label)): 
    if not isinstance(emotion_chunk, str):  # in case of nan
        chunk = music_files['neutral'][:, int(start / eeg_freq * target_freq): int(end / eeg_freq * target_freq)]
        torchaudio.save(os.path.join(*[root_dir, subID, f"music/{data_type}/melody_chunks/chunk{i}.wav"]), chunk, target_freq)
    else: 
        for emotion in emotion_labels: 
            if emotion_chunk == emotion: 
                chunk = music_files[emotion][:, int(start / eeg_freq * target_freq): int(end / eeg_freq * target_freq)]
                torchaudio.save(os.path.join(*[root_dir, subID, f"music/{data_type}/melody_chunks/chunk{i}.wav"]), chunk, target_freq)

In [ ]:
# loading every music and applying cross fading 
for i in range(len(glob.glob(os.path.join(*[root_dir, subID, f"music/{data_type}/melody_chunks/*"])))):
    if i == 0: 
        mixed_song = AudioSegment.from_file(os.path.join(*[root_dir, subID, f"music/{data_type}/melody_chunks/chunk{i}.wav"]))
    else: 
        song_tmp = AudioSegment.from_file(os.path.join(*[root_dir, subID, f"music/{data_type}/melody_chunks/chunk{i}.wav"]))
        mixed_song = mixed_song.append(song_tmp, crossfade=40)

In [ ]:
normalized_mixed_song = effects.normalize(mixed_song) 

In [ ]:
normalized_mixed_song.export(os.path.join(*[root_dir, subID, f"music/{data_type}/melody_mixed_final.wav"]), format='wav')